entity_values - spans

In [ ]:
import ast
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd

INPUT_CSV = Path("/content/final_dataset_for_annotation_v1 - final_dataset_for_annotation.csv.csv")
OUTPUT_DIR = Path("step1_outputs")
TEXT_COL = "text"
ENTITIES_COL = "entity_values"
ID_COL = None


def parse_entity_values(value: Any) -> List[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    if isinstance(value, list):
        return [str(x) for x in value if str(x).strip()]

    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if str(x).strip()]
        except Exception:
            pass

    return []


WHITESPACE_RE = re.compile(r"\s+", flags=re.UNICODE)


def normalize_char(ch: str) -> str:
    ch = ch.lower()
    ch = ch.replace("ё", "е")
    ch = (
        ch.replace("—", "-")
        .replace("–", "-")
        .replace("−", "-")
        .replace("‑", "-")
    )
    ch = (
        ch.replace("«", '"')
        .replace("»", '"')
        .replace("“", '"')
        .replace("”", '"')
        .replace("‘", "'")
        .replace("’", "'")
    )
    return ch


def build_normalized_text_with_mapping(text: str) -> Tuple[str, List[int]]:

    normalized_chars: List[str] = []
    normalized_to_original: List[int] = []

    for original_idx, ch in enumerate(text):
        ch = normalize_char(ch)
        if ch.isspace():
            continue
        normalized_chars.append(ch)
        normalized_to_original.append(original_idx)

    return "".join(normalized_chars), normalized_to_original


def normalize_entity(entity: str) -> str:
    chars = []
    for ch in entity:
        ch = normalize_char(ch)
        if ch.isspace():
            continue
        chars.append(ch)
    return "".join(chars)


def find_all_exact_matches(text: str, entity: str) -> List[Tuple[int, int]]:
    matches: List[Tuple[int, int]] = []
    start = 0
    while True:
        pos = text.find(entity, start)
        if pos == -1:
            break
        matches.append((pos, pos + len(entity)))
        start = pos + 1
    return matches


def find_all_normalized_matches(text: str, entity: str) -> List[Tuple[int, int]]:
    normalized_text, norm_to_orig = build_normalized_text_with_mapping(text)
    normalized_entity = normalize_entity(entity)

    if not normalized_entity:
        return []

    matches: List[Tuple[int, int]] = []
    start = 0
    while True:
        pos = normalized_text.find(normalized_entity, start)
        if pos == -1:
            break

        orig_start = norm_to_orig[pos]
        orig_end = norm_to_orig[pos + len(normalized_entity) - 1] + 1
        matches.append((orig_start, orig_end))
        start = pos + 1

    return matches


def span_overlaps_existing(span: Tuple[int, int], spans: List[Tuple[int, int]]) -> bool:
    s1, e1 = span
    for s2, e2 in spans:
        if max(s1, s2) < min(e1, e2):
            return True
    return False


def resolve_entity_span(
    text: str,
    entity: str,
    already_assigned: List[Tuple[int, int]],
) -> Tuple[Optional[Dict[str, Any]], Dict[str, Any]]:

    exact = find_all_exact_matches(text, entity)
    exact_non_overlapping = [m for m in exact if not span_overlaps_existing(m, already_assigned)]

    if len(exact_non_overlapping) == 1:
        s, e = exact_non_overlapping[0]
        return {
            "text": text[s:e],
            "entity_value": entity,
            "start": s,
            "end": e,
            "match_type": "exact",
        }, {
            "status": "matched_exact",
            "n_candidates": 1,
        }

    if len(exact_non_overlapping) > 1:
        return None, {
            "status": "ambiguous_exact",
            "n_candidates": len(exact_non_overlapping),
            "candidates": exact_non_overlapping,
        }

    normalized = find_all_normalized_matches(text, entity)
    normalized_non_overlapping = [m for m in normalized if not span_overlaps_existing(m, already_assigned)]

    normalized_non_overlapping = list(dict.fromkeys(normalized_non_overlapping))

    if len(normalized_non_overlapping) == 1:
        s, e = normalized_non_overlapping[0]
        return {
            "text": text[s:e],
            "entity_value": entity,
            "start": s,
            "end": e,
            "match_type": "normalized",
        }, {
            "status": "matched_normalized",
            "n_candidates": 1,
        }

    if len(normalized_non_overlapping) > 1:
        return None, {
            "status": "ambiguous_normalized",
            "n_candidates": len(normalized_non_overlapping),
            "candidates": normalized_non_overlapping,
        }

    return None, {
        "status": "unmatched",
        "n_candidates": 0,
    }


def convert_row_entities_to_spans(row: pd.Series) -> pd.Series:
    text = str(row[TEXT_COL])
    entities = parse_entity_values(row[ENTITIES_COL])

    spans: List[Dict[str, Any]] = []
    assigned_spans: List[Tuple[int, int]] = []
    review_items: List[Dict[str, Any]] = []

    entities_sorted = sorted(entities, key=lambda x: (-len(x), x))

    for entity in entities_sorted:
        span, info = resolve_entity_span(text, entity, assigned_spans)
        if span is not None:
            spans.append(span)
            assigned_spans.append((span["start"], span["end"]))
        else:
            review_items.append({
                "entity_value": entity,
                **info,
            })

    spans = sorted(spans, key=lambda x: (x["start"], x["end"]))

    expected_count = len(entities)
    matched_count = len(spans)
    unresolved_count = len(review_items)

    if unresolved_count == 0 and matched_count == expected_count:
        row_status = "ok"
    elif matched_count > 0:
        row_status = "partial_review"
    else:
        row_status = "full_review"

    return pd.Series({
        "parsed_entities": entities,
        "entity_spans": spans,
        "matched_entities_count": matched_count,
        "unresolved_entities_count": unresolved_count,
        "row_status": row_status,
        "review_items": review_items,
    })


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(INPUT_CSV)
    required_cols = {"source", TEXT_COL, "n_entities", ENTITIES_COL}
    missing = required_cols - set(df.columns)

    converted = df.apply(convert_row_entities_to_spans, axis=1)
    result_df = pd.concat([df, converted], axis=1)

    result_df["parsed_entities_json"] = result_df["parsed_entities"].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
    )
    result_df["entity_spans_json"] = result_df["entity_spans"].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
    )
    result_df["review_items_json"] = result_df["review_items"].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
    )

    export_cols = [
        "source",
        "text",
        "n_entities",
        "entity_values",
        "matched_entities_count",
        "unresolved_entities_count",
        "row_status",
        "parsed_entities_json",
        "entity_spans_json",
        "review_items_json",
    ]
    result_df[export_cols].to_csv(OUTPUT_DIR / "dataset_with_spans.csv", index=False)

    review_df = result_df[result_df["row_status"] != "ok"].copy()
    review_df[export_cols].to_csv(OUTPUT_DIR / "rows_for_manual_review.csv", index=False)

    review_records: List[Dict[str, Any]] = []
    for row_idx, row in result_df.iterrows():
        for item in row["review_items"]:
            review_records.append({
                "row_index": row_idx,
                "source": row["source"],
                "text": row["text"],
                "entity_values": row["entity_values"],
                "entity_value": item.get("entity_value"),
                "status": item.get("status"),
                "n_candidates": item.get("n_candidates"),
                "candidates": json.dumps(item.get("candidates", []), ensure_ascii=False),
            })

    pd.DataFrame(review_records).to_csv(OUTPUT_DIR / "problem_entities_flat.csv", index=False)

    stats = {
        "n_rows": int(len(result_df)),
        "n_ok_rows": int((result_df["row_status"] == "ok").sum()),
        "n_partial_review_rows": int((result_df["row_status"] == "partial_review").sum()),
        "n_full_review_rows": int((result_df["row_status"] == "full_review").sum()),
        "n_total_entities": int(result_df["parsed_entities"].apply(len).sum()),
        "n_matched_entities": int(result_df["matched_entities_count"].sum()),
        "n_unresolved_entities": int(result_df["unresolved_entities_count"].sum()),
        "match_rate": round(
            float(result_df["matched_entities_count"].sum())
            / max(1, float(result_df["parsed_entities"].apply(len).sum())),
            4,
        ),
    }

    with open(OUTPUT_DIR / "stats.json", "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    print(json.dumps(stats, ensure_ascii=False, indent=2))
    print("- dataset_with_spans.csv")
    print("- rows_for_manual_review.csv")
    print("- problem_entities_flat.csv")
    print("- stats.json")


if __name__ == "__main__":
    main()


In [ ]:

import ast
import json
import os
import re
from typing import Any, Dict, List, Tuple

import pandas as pd

INPUT_PATH = "/content/step1_outputs/dataset_with_spans.csv"
OUTPUT_DIR = "/content/sample_data/step2_outputs"
OUTPUT_DATASET = os.path.join(OUTPUT_DIR, "dataset_with_typed_spans_v2.csv")
OUTPUT_ROWS_REVIEW = os.path.join(OUTPUT_DIR, "rows_for_type_review_v2.csv")
OUTPUT_ITEMS_REVIEW = os.path.join(OUTPUT_DIR, "type_review_items_flat_v2.csv")
OUTPUT_STATS = os.path.join(OUTPUT_DIR, "type_stats_v2.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)

NAME_PARTICLES = {
    "оглы", "кызы", "ибн", "бин", "бен", "аль", "эль", "де", "дел", "ла", "ле", "фон", "ван", "да", "ди"
}

PERSON_ROLE_WORDS = {
    "президент", "министр", "директор", "генеральный", "секретарь", "руководитель",
    "истец", "ответчик", "свидетель", "эксперт", "подсудимый", "заявитель",
    "гражданин", "гр", "гр.", "физлицо", "ип", "адвокат", "судья", "доктор",
    "профессор", "господин", "товарищ", "коллега", "мать", "отец", "сын", "дочь",
    "бабушка", "дедушка", "брат", "сестра"
}

ADDRESS_MARKERS = {
    "г", "г.", "город", "обл", "обл.", "область", "край", "респ", "респ.", "республика", "ао",
    "район", "р-н", "пос", "пос.", "поселок", "пгт", "д", "д.", "дер", "дер.", "деревня",
    "с", "с.", "село", "х", "х.", "хутор", "ул", "ул.", "улица", "пр-кт", "проспект", "пр.",
    "пер", "пер.", "переулок", "наб", "наб.", "набережная", "б-р", "бул", "бул.", "бульвар",
    "ш", "ш.", "шоссе", "тракт", "дом", "стр", "стр.", "строение", "корп", "корп.", "корпус",
    "к", "к.", "кв", "кв.", "квартира", "оф", "оф.", "офис", "вл", "вл.", "владение",
    "лит", "лит.", "этаж", "эт", "эт.", "уч", "уч.", "участок", "в/ч", "общежитие",
    "км", "литер", "округ"
}

PHONE_CONTEXT = {
    "тел", "тел.", "телефон", "телефоны", "контактный", "номер", "моб", "моб.",
    "звонить", "наберите", "связаться", "связи", "доб", "доб.", "добавочный",
    "внутренний", "вн", "вн.", "ext", "#"
}

ID_CONTEXT = {
    "id", "id:", "id=", "user_id", "user_id:", "user_id=", "номер", "№",
    "договор", "контракт", "заявка", "инн", "снилс", "счет", "счёт",
    "р/с", "клиента", "договора", "номер заявки", "идентификатор"
}

EMAIL_CONTEXT = {"email", "e-mail", "почта", "электронная", "эл.", "mail"}

RU_NUM_WORDS = {
    "ноль", "один", "одна", "два", "две", "три", "четыре", "пять", "шесть",
    "семь", "восемь", "девять", "десять", "двадцать", "тридцать", "сорок",
    "пятьдесят", "шестьдесят", "семьдесят", "восемьдесят", "девяносто",
    "сто", "двести", "триста", "четыреста", "пятьсот", "шестьсот",
    "семьсот", "восемьсот", "девятьсот", "тысяч"
}

WRAP_PUNCT = " \t\n\r\"'“”«»[](){}<>.,;:!?…"

EMAIL_RE = re.compile(
    r"(?iu)(?:[a-z0-9а-яё][a-z0-9а-яё._%+\-]*)\s*@\s*"
    r"(?:[a-z0-9а-яё\-]+\s*(?:\.\s*[a-z0-9а-яё\-]+)*|localhost|\d{1,3}(?:\.\d{1,3}){3})"
)

SNILS_RE = re.compile(r"^\s*\d{3}-?\d{3}-?\d{3}\s?\d{2}\s*$")
INN_RE = re.compile(r"^\s*\d{10}(\d{2})?\s*$")
BANK_RE = re.compile(r"^\s*\d{20}\s*$")
INITIALS_TOKEN_RE = re.compile(r"^[А-ЯЁA-Z]\.?([А-ЯЁA-Z]\.?)?$")
CAPITAL_WORD_RE = re.compile(r"^[А-ЯЁA-Z][а-яёa-z]+(?:-[А-ЯЁA-Z]?[а-яёa-z]+)*$")
FULL_ID_KEY_RE = re.compile(r"(?iu)\b(?:id|user_id)\s*[:=]")
POSTAL_RE = re.compile(r"\b\d{5,6}\b")
PHONE_DIGITS_RE = re.compile(r"\d")

def norm(s: Any) -> str:
    if s is None:
        return ""
    s = str(s)
    repl = {
        "\u00A0": " ",
        "\u2009": " ",
        "\u202f": " ",
        "\t": " ",
        "\r": " ",
        "\n": " ",
        "—": "-",
        "–": "-",
        "−": "-",
        "‑": "-",
        "“": '"',
        "”": '"',
        "«": '"',
        "»": '"',
    }
    for a, b in repl.items():
        s = s.replace(a, b)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def strip_wrappers(s: str) -> str:
    return s.strip(WRAP_PUNCT)


def compact_email_spaces(s: str) -> str:
    s = re.sub(r"\s*@\s*", "@", s)
    s = re.sub(r"\s*\.\s*", ".", s)
    return s


def numeric_word_count(s: str) -> int:
    toks = re.findall(r"(?iu)[а-яёa-z]+", s.lower())
    return sum(t in RU_NUM_WORDS for t in toks)


def get_context(text: str, start: int, end: int, window: int = 40) -> str:
    fragment = text[max(0, start - window): min(len(text), end + window)]
    return norm(fragment).lower()


def cleaned_candidate(span_text: str) -> str:
    s = norm(span_text)
    s = re.sub(
        r"(?iu)^(?:электронная\s+почта|email|e-mail|почта|"
        r"телефон|тел\.?|контактный\s+телефон|контактный\s+номер|номер\s+телефона|"
        r"идентификатор|id|user_id|адрес|место\s+проживания)\s*[:=]\s*",
        "",
        s,
    )
    s = strip_wrappers(s)
    return s


def safe_json_loads(value: Any) -> Any:
    if isinstance(value, (list, dict)):
        return value
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    text = str(value).strip()
    if not text:
        return []
    try:
        return json.loads(text)
    except Exception:
        try:
            return ast.literal_eval(text)
        except Exception:
            return []


def is_email(candidate: str, context: str) -> Tuple[bool, str]:
    c = compact_email_spaces(candidate)
    if EMAIL_RE.search(c):
        return True, "regex_email"
    if any(k in context for k in EMAIL_CONTEXT) and "@" in c:
        return True, "context_email"
    return False, ""


def is_address(candidate: str, context: str) -> Tuple[bool, str]:
    c = candidate
    low = c.lower()
    toks = re.findall(r"(?iu)[a-zа-яё0-9/-]+\.?", low)
    marker_hits = [t for t in toks if t in ADDRESS_MARKERS]
    has_ctx = "адрес" in context or "проживания" in context or "место проживания" in context

    if len(marker_hits) >= 2:
        return True, "marker_count_address"

    if POSTAL_RE.search(c) and (len(marker_hits) >= 1 or has_ctx):
        return True, "postal_address"

    if has_ctx and (
        len(marker_hits) >= 1
        or re.search(r"(?iu)\b(?:москва|санкт-петербург|спб|казань|самара|уфа|россия)\b", low)
    ):
        return True, "context_address"

    if re.search(
        r"(?iu)\b(?:г\.?\s*[А-ЯЁA-Z][а-яёa-z-]+|"
        r"ул\.?\s*[А-ЯЁA-ZA-Za-z][^,;]{0,30}|"
        r"д\.?\s*\d+[а-яёa-z]?|"
        r"кв\.?\s*\d+[а-яёa-z]?)",
        c,
    ):
        return True, "pattern_address"

    if re.search(r"(?iu)\bв/ч\s*\d+\b", low):
        return True, "military_address"

    if re.search(r"(?iu)\bкм\b", low) and re.search(r"\d", c) and ("трасс" in low or "тракт" in low):
        return True, "special_address"

    if any(h in low for h in ["street", "apt", "new york", "usa", "russia", "moscow"]) and re.search(r"\d", c):
        return True, "intl_address"

    return False, ""


def is_id(candidate: str, context: str) -> Tuple[bool, str]:
    c = candidate
    low = c.lower()
    digits = re.sub(r"\D", "", c)

    if SNILS_RE.match(c):
        return True, "snils"

    if BANK_RE.match(digits):
        return True, "bank_account"

    if "инн" in low or INN_RE.match(digits):
        return True, "inn"

    if FULL_ID_KEY_RE.search(low):
        return True, "id_key"

    if "№" in c or "#" in c:
        if re.search(r"(?iu)[a-zа-я0-9/-]{2,}", c):
            return True, "num_sign_id"

    if any(k in context for k in ["договор", "контракт", "заявка", "счет", "счёт", "идентификатор", "клиента"]) and re.search(r"\d", c):
        return True, "context_id"

    if re.search(r"(?iu)\b(?:договор|контракт|заявка)\b", low):
        return True, "doc_id"

    if 5 <= len(digits) <= 9:
        if any(k in context for k in ["id", "номер заявки", "заявка", "договор", "контракт", "клиента", "счет", "счёт"]):
            return True, "short_context_id"

    if re.fullmatch(r"(?iu)[a-zа-я0-9/_=\-]{5,}", low) and re.search(r"\d", low) and "@" not in low:
        return True, "alnum_id"

    return False, ""


def is_phone(candidate: str, context: str) -> Tuple[bool, str]:
    c = candidate
    low = c.lower()
    digits = re.sub(r"\D", "", c)
    has_phone_ctx = any(k in context for k in PHONE_CONTEXT) or re.search(r"(?iu)\b(?:тел|телефон|моб|доб|вн|ext)\b", low) is not None

    allowed_words = {
        "тел", "телефон", "контактный", "номер", "моб", "доб", "добавочный",
        "внутренний", "вн", "ext", "по", "связаться", "звонить", "наберите"
    }
    letter_tokens = re.findall(r"(?iu)[a-zа-яё]+", low)
    bad_letter_tokens = [t for t in letter_tokens if t not in allowed_words and t not in RU_NUM_WORDS]

    if numeric_word_count(c) >= 3 and has_phone_ctx:
        return True, "word_number_phone"

    if bad_letter_tokens and not has_phone_ctx:
        return False, ""

    if 10 <= len(digits) <= 12:
        if SNILS_RE.match(c) or BANK_RE.match(digits) or "@" in c:
            return False, ""
        if (
            re.search(r"[+().\- ]", c)
            or has_phone_ctx
            or c.strip().startswith(("7", "8", "+7"))
            or re.fullmatch(r"\d{10,12}", digits)
        ):
            return True, "digits10_12_phone"

    if 5 <= len(digits) <= 9:
        if has_phone_ctx:
            return True, "context_short_phone"
        if re.fullmatch(r"\d{1,3}[- ]\d{2}[- ]\d{2}", c) or re.fullmatch(r"\d{2,3}[- ]\d{2}[- ]\d{2}", c):
            return True, "local_phone"
        if re.fullmatch(r"\(\d{3}\)\s*\d{3}[- ]\d{2}[- ]\d{2}", c):
            return True, "code_phone"

    return False, ""


def looks_like_person_token(tok: str) -> bool:
    t = tok.strip(".,")
    low = t.lower()
    if low in NAME_PARTICLES:
        return True
    if INITIALS_TOKEN_RE.match(t):
        return True
    if CAPITAL_WORD_RE.match(t):
        return True
    return False


def is_person(candidate: str, context: str) -> Tuple[bool, str]:
    c = candidate
    c2 = re.sub(
        r"(?iu)^(?:(?:гр\.?|гражданин|физлицо|заявитель|подсудимый|ип|истец|свидетель|эксперт|"
        r"мать|отец|сын|дочь|бабушка|дедушка|брат|сестра|"
        r"президент|министр|директор|руководитель)\s+)+",
        "",
        c,
    ).strip(" ,")
    tokens = [t for t in re.split(r"\s+", c2) if t]
    if not tokens:
        return False, ""

    if "@" in c2 or re.search(r"\d{3,}", c2):
        return False, ""

    good = 0
    for tok in tokens:
        if looks_like_person_token(tok):
            good += 1
        elif tok.lower() in PERSON_ROLE_WORDS:
            continue

    if good >= 1 and good >= max(1, len(tokens) - 1):
        if 1 <= len(tokens) <= 6:
            return True, "name_like"

    if len(tokens) == 1 and CAPITAL_WORD_RE.match(tokens[0]) and any(k in context for k in PERSON_ROLE_WORDS):
        return True, "context_person_single"

    return False, ""


def classify_span(span_text: str, full_text: str, start: int, end: int) -> Tuple[str, str, str, str]:
    candidate = cleaned_candidate(span_text)
    context = get_context(full_text, start, end)

    checks = [
        ("EMAIL", is_email),
        ("ADDRESS", is_address),
        ("ID", is_id),
        ("PHONE", is_phone),
        ("PERSON", is_person),
    ]

    for label, fn in checks:
        ok, method = fn(candidate, context)
        if ok:
            if method in {
                "regex_email", "marker_count_address", "postal_address", "snils",
                "bank_account", "inn", "digits10_12_phone"
            }:
                confidence = "high"
            else:
                confidence = "medium"
            return label, method, confidence, candidate

    return "UNKNOWN", "no_rule", "low", candidate


def main() -> None:
    df = pd.read_csv(INPUT_PATH)

    typed_spans_json_list: List[str] = []
    rows_for_review: List[Dict[str, Any]] = []
    review_items_flat: List[Dict[str, Any]] = []

    total_spans = 0
    type_counts: Dict[str, int] = {"PERSON": 0, "PHONE": 0, "EMAIL": 0, "ADDRESS": 0, "ID": 0, "UNKNOWN": 0}
    method_counts: Dict[str, int] = {}

    for idx, row in df.iterrows():
        text = row["text"]
        spans = safe_json_loads(row.get("entity_spans_json", "[]"))
        typed_spans = []
        row_review_items = []

        for span in spans:
            span_text = span.get("text", "")
            start = int(span.get("start", -1))
            end = int(span.get("end", -1))

            label, method, confidence, normalized_candidate = classify_span(span_text, text, start, end)

            out = dict(span)
            out["entity_type"] = label
            out["type_method"] = method
            out["type_confidence"] = confidence
            out["normalized_candidate"] = normalized_candidate
            typed_spans.append(out)

            total_spans += 1
            type_counts[label] = type_counts.get(label, 0) + 1
            method_counts[method] = method_counts.get(method, 0) + 1

            if label == "UNKNOWN" or confidence == "low":
                item = {
                    "row_index": int(idx),
                    "source": row.get("source"),
                    "text": text,
                    "span_text": span_text,
                    "start": start,
                    "end": end,
                    "normalized_candidate": normalized_candidate,
                    "predicted_type": label,
                    "type_method": method,
                    "type_confidence": confidence,
                }
                row_review_items.append(item)
                review_items_flat.append(item)

        typed_spans_json_list.append(json.dumps(typed_spans, ensure_ascii=False))

        if row_review_items:
            rows_for_review.append(
                {
                    "row_index": int(idx),
                    "source": row.get("source"),
                    "text": text,
                    "typed_spans_json": json.dumps(typed_spans, ensure_ascii=False),
                    "review_items_json": json.dumps(row_review_items, ensure_ascii=False),
                }
            )

    df["typed_entity_spans_json"] = typed_spans_json_list
    df["typed_spans_count"] = df["typed_entity_spans_json"].apply(lambda s: len(safe_json_loads(s)))
    df["unknown_typed_spans_count"] = df["typed_entity_spans_json"].apply(
        lambda s: sum(1 for x in safe_json_loads(s) if x.get("entity_type") == "UNKNOWN")
    )
    df["type_row_status"] = df["unknown_typed_spans_count"].apply(lambda n: "needs_review" if n > 0 else "ok")

    rows_review_df = pd.DataFrame(rows_for_review)
    items_review_df = pd.DataFrame(review_items_flat)

    stats = {
        "input_path": INPUT_PATH,
        "rows_total": int(len(df)),
        "spans_total": int(total_spans),
        "type_counts": type_counts,
        "method_counts": method_counts,
        "rows_needing_type_review": int(len(rows_review_df)),
        "unknown_total": int(type_counts.get("UNKNOWN", 0)),
    }

    df.to_csv(OUTPUT_DATASET, index=False)
    rows_review_df.to_csv(OUTPUT_ROWS_REVIEW, index=False)
    items_review_df.to_csv(OUTPUT_ITEMS_REVIEW, index=False)

    with open(OUTPUT_STATS, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    print(json.dumps(stats, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


In [ ]:

import json
import re
from pathlib import Path
import pandas as pd

INPUT_CSV = Path("/content/sample_data/step2_outputs/dataset_with_typed_spans_v2.csv")
OUTPUT_DIR = Path("/content/sample_data/step3_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "dataset_with_regex_augmented.csv"
NEW_ITEMS_FLAT_CSV = OUTPUT_DIR / "regex_new_items_flat.csv"
CONFLICTS_FLAT_CSV = OUTPUT_DIR / "regex_conflicts_flat.csv"
STATS_JSON = OUTPUT_DIR / "regex_stats.json"


def parse_json_list(value):
    if pd.isna(value) or value == "":
        return []
    if isinstance(value, list):
        return value
    return json.loads(value)


def norm_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()


def trim_punct(text: str, start: int, end: int):
    while start < end and text[start] in ' \t\n\r"\'([{<':
        start += 1
    while end > start and text[end - 1] in ' \t\n\r"\')]}>.,;:!?':
        end -= 1
    return start, end


EMAIL_PATTERNS = [
    re.compile(
        r'(?P<full>'
        r'(?:email|e-mail|почта|электронная почта)\s*[:=]\s*'
        r'(?P<addr>[A-Za-zА-Яа-яЁё0-9._%+\-]+(?:\s*@\s*|\@)'
        r'(?:[A-Za-zА-Яа-яЁё0-9\-]+(?:\s*\.\s*[A-Za-zА-Яа-яЁё0-9\-]+)+|localhost|\d{1,3}(?:\.\d{1,3}){3}))'
        r')',
        flags=re.IGNORECASE
    ),
    re.compile(
        r'(?P<addr>[A-Za-zА-Яа-яЁё0-9._%+\-]+(?:\s*@\s*|\@)'
        r'(?:[A-Za-zА-Яа-яЁё0-9\-]+(?:\s*\.\s*[A-Za-zА-Яа-яЁё0-9\-]+)+|localhost|\d{1,3}(?:\.\d{1,3}){3}))',
        flags=re.IGNORECASE
    ),
]

PHONE_CONTEXT_PREFIX = r'(?:тел\.?|телефон|контактный номер|моб\.?|мобильный|номер телефона|звонить по номеру|наберите|связаться по)\s*[:=]?\s*'
PHONE_MAIN = (
    r'(?:\+?\s*7|8)?\s*'
    r'(?:\(\s*\d{3}\s*\)|\d{3})?'
    r'(?:[\s.\-]*\d){5,10}'
    r'(?:\s*(?:доб\.?|добавочный|вн\.?|внутренний|ext|#)\s*\d{1,5})?'
)
PHONE_PATTERNS = [
    re.compile(rf'(?P<full>{PHONE_CONTEXT_PREFIX}(?P<num>{PHONE_MAIN}))', flags=re.IGNORECASE),
    re.compile(rf'(?P<num>{PHONE_MAIN})', flags=re.IGNORECASE),
    re.compile(r'(?P<num>\d{1,3}(?:[\s\-]\d{2}){2,3})'),
]

ID_PATTERNS = [
    re.compile(r'(?P<id>(?:ID|id|user_id|id клиента)\s*[:=]\s*[A-Za-zА-Яа-яЁё0-9\-_/]+)'),
    re.compile(r'(?P<id>(?:договор|контракт|заявка|номер заявки|счет|сч[её]т|р/с)\s*№?\s*[A-Za-zА-Яа-яЁё0-9\-_/]+)', flags=re.IGNORECASE),
    re.compile(r'(?P<id>№\s*[A-Za-zА-Яа-яЁё0-9\-_/]+)'),
    re.compile(r'(?P<id>ИНН\s*:?\s*\d{10,12})', flags=re.IGNORECASE),
    re.compile(r'(?P<id>\d{3}-\d{3}-\d{3}\s?\d{2})'),
    re.compile(r'(?P<id>\b\d{11}\b)'),
    re.compile(r'(?P<id>\b4\d{19}\b)'),
    re.compile(r'(?P<id>\b\d{20}\b)'),
    re.compile(r'(?P<id>\b\d{10,12}\b)'),
]


ADDRESS_HINTS = re.compile(
    r'(?:\bг\.?\s*[А-ЯA-ZЁ][а-яa-zё\-]+|\bул\.?\s*[А-ЯA-ZЁ][а-яa-zё\-]+|\bд\.?\s*\d+[А-ЯA-Zа-яa-z]?)'
)


def clean_email(candidate: str) -> str:
    c = candidate.strip()
    c = re.sub(r'^[\(\[\{<"\']+', '', c)
    c = re.sub(r'[\)\]\}>,"\';:!?]+$', '', c)
    c = re.sub(r'\s*@\s*', '@', c)
    c = re.sub(r'\s*\.\s*', '.', c)
    return c


def is_valid_email(candidate: str) -> bool:
    c = clean_email(candidate)
    return bool(re.fullmatch(
        r'[A-Za-zА-Яа-яЁё0-9._%+\-]+@(?:[A-Za-zА-Яа-яЁё0-9\-]+(?:\.[A-Za-zА-Яа-яЁё0-9\-]+)+|localhost|\d{1,3}(?:\.\d{1,3}){3})',
        c
    ))


def clean_phone(candidate: str) -> str:
    c = candidate.strip()
    c = re.sub(rf'^(?:{PHONE_CONTEXT_PREFIX})', '', c, flags=re.IGNORECASE)
    return norm_spaces(c)


def is_phone_candidate(candidate: str) -> bool:
    c = clean_phone(candidate)
    digits = re.sub(r'\D', '', c)
    if 'доб' in c.lower() or 'вн' in c.lower() or 'ext' in c.lower() or '#' in c:
        return len(digits) >= 7
    return 5 <= len(digits) <= 12


def looks_like_id(candidate: str) -> bool:
    c = candidate.strip()
    lower = c.lower()
    digits = re.sub(r'\D', '', c)
    if re.search(r'(id|user_id|инн|снилс|договор|контракт|заявк|счет|счёт|р/с|№)', lower):
        return True
    if re.fullmatch(r'\d{20}', digits):
        return True
    if re.fullmatch(r'\d{10,12}', digits):
        return True
    if re.fullmatch(r'\d{11}', digits):
        return True
    if re.fullmatch(r'[A-Za-zА-Яа-яЁё0-9\-_/]+', c) and any(ch.isdigit() for ch in c):
        return True
    return False


def span_overlaps(a_start, a_end, b_start, b_end):
    return not (a_end <= b_start or b_end <= a_start)


def overlap_ratio(a_start, a_end, b_start, b_end):
    inter = max(0, min(a_end, b_end) - max(a_start, b_start))
    union = max(a_end, b_end) - min(a_start, b_start)
    return inter / union if union else 0.0


def detect_email_spans(text: str):
    found = []
    for pattern in EMAIL_PATTERNS:
        for m in pattern.finditer(text):
            if 'addr' in m.groupdict() and m.group('addr'):
                start, end = m.start('addr'), m.end('addr')
            else:
                start, end = m.start(), m.end()
            start, end = trim_punct(text, start, end)
            candidate = text[start:end]
            if is_valid_email(candidate):
                found.append({
                    "start": start,
                    "end": end,
                    "text": clean_email(candidate),
                    "entity_type": "EMAIL",
                    "regex_method": "regex_email",
                    "regex_confidence": "high",
                })
    return dedup_spans(found)


def detect_phone_spans(text: str):
    found = []
    for pattern in PHONE_PATTERNS:
        for m in pattern.finditer(text):
            if 'num' in m.groupdict() and m.group('num'):
                start, end = m.start('num'), m.end('num')
            else:
                start, end = m.start(), m.end()
            start, end = trim_punct(text, start, end)
            candidate = text[start:end]
            if is_phone_candidate(candidate):
                if '@' in candidate:
                    continue
                digits = re.sub(r'\D', '', candidate)
                if len(digits) >= 13:
                    continue
                found.append({
                    "start": start,
                    "end": end,
                    "text": norm_spaces(candidate),
                    "entity_type": "PHONE",
                    "regex_method": "regex_phone",
                    "regex_confidence": "high" if len(digits) >= 10 else "medium",
                })
    return dedup_spans(found)


def detect_id_spans(text: str):
    found = []
    for pattern in ID_PATTERNS:
        for m in pattern.finditer(text):
            start, end = m.start('id') if 'id' in m.groupdict() else m.start(), m.end('id') if 'id' in m.groupdict() else m.end()
            start, end = trim_punct(text, start, end)
            candidate = text[start:end]
            if looks_like_id(candidate):
                lower_ctx = text[max(0, start - 20):min(len(text), end + 20)].lower()
                if re.search(r'(тел\.?|телефон|контактный номер|моб\.?)', lower_ctx):
                    digits = re.sub(r'\D', '', candidate)
                    if 5 <= len(digits) <= 12:
                        continue
                if ADDRESS_HINTS.search(candidate):
                    continue
                found.append({
                    "start": start,
                    "end": end,
                    "text": norm_spaces(candidate),
                    "entity_type": "ID",
                    "regex_method": "regex_id",
                    "regex_confidence": "high",
                })
    return dedup_spans(found)


def dedup_spans(spans):
    if not spans:
        return []
    spans_sorted = sorted(spans, key=lambda x: (x["start"], x["end"], x["entity_type"]))
    result = []
    seen = set()
    for sp in spans_sorted:
        key = (sp["start"], sp["end"], sp["entity_type"], sp["text"])
        if key in seen:
            continue
        seen.add(key)
        result.append(sp)
    return result


def merge_regex_with_existing(text, existing_spans, regex_spans):
    merged = [dict(x) for x in existing_spans]
    new_items = []
    conflicts = []

    for rgx in regex_spans:
        overlaps = [ex for ex in merged if span_overlaps(rgx["start"], rgx["end"], ex["start"], ex["end"])]
        if not overlaps:
            new_item = {
                "text": text[rgx["start"]:rgx["end"]],
                "entity_value": text[rgx["start"]:rgx["end"]],
                "start": rgx["start"],
                "end": rgx["end"],
                "match_type": "regex_added",
                "entity_type": rgx["entity_type"],
                "type_method": rgx["regex_method"],
                "type_confidence": rgx["regex_confidence"],
                "normalized_candidate": rgx["text"],
                "origin": "step3_regex",
            }
            merged.append(new_item)
            new_items.append(new_item)
            continue

        same_type_exact = [
            ex for ex in overlaps
            if ex.get("entity_type") == rgx["entity_type"]
            and ex["start"] == rgx["start"]
            and ex["end"] == rgx["end"]
        ]
        if same_type_exact:
            continue

        upgraded = False
        for ex in overlaps:
            if ex.get("entity_type") == "UNKNOWN" and overlap_ratio(rgx["start"], rgx["end"], ex["start"], ex["end"]) >= 0.7:
                ex["entity_type"] = rgx["entity_type"]
                ex["type_method"] = rgx["regex_method"] + "_upgrade_unknown"
                ex["type_confidence"] = rgx["regex_confidence"]
                ex["normalized_candidate"] = rgx["text"]
                ex["origin"] = "step2_plus_step3_regex_upgrade"
                upgraded = True
                conflicts.append({
                    "action": "upgrade_unknown",
                    "regex_span": rgx,
                    "existing_span": ex,
                })
                break
        if upgraded:
            continue

        conflicts.append({
            "action": "overlap_conflict",
            "regex_span": rgx,
            "existing_spans": overlaps,
        })

    merged = sorted(merged, key=lambda x: (x["start"], x["end"]))
    return merged, new_items, conflicts


df = pd.read_csv(INPUT_CSV)

all_new_flat = []
all_conflicts_flat = []

total_regex_found = 0
total_new_added = 0
total_unknown_upgraded = 0
rows_with_new = 0
rows_with_conflicts = 0

augmented_json_col = []
regex_new_count_col = []
regex_conflict_count_col = []
regex_new_json_col = []
regex_conflicts_json_col = []

for idx, row in df.iterrows():
    text = row["text"]
    existing = parse_json_list(row["typed_entity_spans_json"])

    email_spans = detect_email_spans(text)
    phone_spans = detect_phone_spans(text)
    id_spans = detect_id_spans(text)

    regex_spans = dedup_spans(email_spans + phone_spans + id_spans)
    total_regex_found += len(regex_spans)

    merged, new_items, conflicts = merge_regex_with_existing(text, existing, regex_spans)

    rows_with_new += int(bool(new_items))
    rows_with_conflicts += int(bool(conflicts))
    total_new_added += len(new_items)
    total_unknown_upgraded += sum(1 for c in conflicts if c.get("action") == "upgrade_unknown")

    augmented_json_col.append(json.dumps(merged, ensure_ascii=False))
    regex_new_count_col.append(len(new_items))
    regex_conflict_count_col.append(len(conflicts))
    regex_new_json_col.append(json.dumps(new_items, ensure_ascii=False))
    regex_conflicts_json_col.append(json.dumps(conflicts, ensure_ascii=False))

    for item in new_items:
        all_new_flat.append({
            "row_id": idx,
            "source": row["source"],
            "text": text,
            **item
        })

    for c in conflicts:
        if c["action"] == "upgrade_unknown":
            all_conflicts_flat.append({
                "row_id": idx,
                "source": row["source"],
                "text": text,
                "action": c["action"],
                "regex_text": c["regex_span"]["text"],
                "regex_type": c["regex_span"]["entity_type"],
                "regex_start": c["regex_span"]["start"],
                "regex_end": c["regex_span"]["end"],
                "existing_text": c["existing_span"]["text"],
                "existing_type_before": "UNKNOWN",
                "existing_start": c["existing_span"]["start"],
                "existing_end": c["existing_span"]["end"],
            })
        else:
            overlaps_serialized = json.dumps(c["existing_spans"], ensure_ascii=False)
            all_conflicts_flat.append({
                "row_id": idx,
                "source": row["source"],
                "text": text,
                "action": c["action"],
                "regex_text": c["regex_span"]["text"],
                "regex_type": c["regex_span"]["entity_type"],
                "regex_start": c["regex_span"]["start"],
                "regex_end": c["regex_span"]["end"],
                "existing_overlaps_json": overlaps_serialized,
            })

df["typed_entity_spans_regex_augmented_json"] = augmented_json_col
df["regex_new_count"] = regex_new_count_col
df["regex_conflict_count"] = regex_conflict_count_col
df["regex_new_items_json"] = regex_new_json_col
df["regex_conflicts_json"] = regex_conflicts_json_col

type_counts = {"PERSON": 0, "PHONE": 0, "EMAIL": 0, "ADDRESS": 0, "ID": 0, "UNKNOWN": 0}
for items_json in df["typed_entity_spans_regex_augmented_json"]:
    items = parse_json_list(items_json)
    for it in items:
        et = it.get("entity_type", "UNKNOWN")
        type_counts[et] = type_counts.get(et, 0) + 1

df.to_csv(OUTPUT_CSV, index=False)
pd.DataFrame(all_new_flat).to_csv(NEW_ITEMS_FLAT_CSV, index=False)
pd.DataFrame(all_conflicts_flat).to_csv(CONFLICTS_FLAT_CSV, index=False)

stats = {
    "input_rows": int(len(df)),
    "total_regex_found_candidates": int(total_regex_found),
    "total_new_added": int(total_new_added),
    "total_unknown_upgraded": int(total_unknown_upgraded),
    "rows_with_new": int(rows_with_new),
    "rows_with_conflicts": int(rows_with_conflicts),
    "final_type_counts_after_regex": type_counts,
}

STATS_JSON.write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps(stats, ensure_ascii=False, indent=2))


Преобразование в формат Doccano

In [ ]:
import pandas as pd
import json

INPUT_PATH = "/content/sample_data/step3_outputs/dataset_with_regex_augmented.csv"
OUTPUT_PATH = "doccano_ready.jsonl"

df = pd.read_csv(INPUT_PATH)

def safe_load(x):
    try:
        return json.loads(x.replace("'", '"'))
    except:
        return []

def convert_row(row):
    text = row["text"]
    spans = safe_load(row["typed_entity_spans_regex_augmented_json"])

    labels = []

    for s in spans:
        start = s["start"]
        end = s["end"]
        label = s["entity_type"]

        if label == "UNKNOWN":
            continue

        if start >= end:
            continue
        if end > len(text):
            continue

        labels.append([start, end, label])

    return {
        "text": text,
        "labels": labels
    }

records = df.apply(convert_row, axis=1).tolist()

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(OUTPUT_PATH)

Разделение на тест (300 текстов)

In [ ]:
import pandas as pd

INPUT_FILE = "/content/шаг_3_dataset_with_regex_augmented - dataset_with_regex_augmented.csv.csv"
OUTPUT_FILE = "dataset_grouped_by_n_entities_then_source.csv"

df = pd.read_csv(INPUT_FILE)


required_cols = {"source", "n_entities"}
missing = required_cols - set(df.columns)

df.insert(0, "original_row_id", range(len(df)))

df["n_entities"] = pd.to_numeric(df["n_entities"], errors="coerce")

df["has_valid_n_entities"] = df["n_entities"].notna()

df.loc[df["has_valid_n_entities"], "n_entities"] = (
    df.loc[df["has_valid_n_entities"], "n_entities"].astype(int)
)

df["group_n_entities"] = df["n_entities"]
df["group_source"] = df["source"].fillna("NaN")

df_sorted = df.sort_values(
    by=["has_valid_n_entities", "group_n_entities", "group_source", "original_row_id"],
    ascending=[False, True, True, True]
).reset_index(drop=True)

df_sorted["group_id"] = (
    df_sorted.groupby(["group_n_entities", "group_source"], dropna=False).ngroup() + 1
)

df_sorted["row_number_within_group"] = (
    df_sorted.groupby(["group_n_entities", "group_source"], dropna=False).cumcount() + 1
)

group_sizes = (
    df_sorted.groupby(["group_n_entities", "group_source"], dropna=False)
    .size()
    .reset_index(name="group_size")
)

df_sorted = df_sorted.merge(
    group_sizes,
    on=["group_n_entities", "group_source"],
    how="left"
)

df_sorted.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd

INPUT_FILE = "/content/шаг_3_dataset_with_regex_augmented - dataset_with_regex_augmented.csv.csv"

df = pd.read_csv(INPUT_FILE)

required_cols = {"source", "n_entities"}
missing = required_cols - set(df.columns)

df["n_entities"] = pd.to_numeric(df["n_entities"], errors="coerce")

df_valid = df[df["n_entities"].notna()].copy()
df_valid["n_entities"] = df_valid["n_entities"].astype(int)

group_counts = (
    df_valid.groupby("n_entities")
    .size()
    .reset_index(name="total_texts")
    .sort_values("n_entities")
)

print(group_counts.to_string(index=False))

group_source_counts = (
    df_valid.groupby(["n_entities", "source"])
    .size()
    .reset_index(name="count_texts")
    .sort_values(["n_entities", "count_texts", "source"], ascending=[True, False, True])
)

print(group_source_counts.to_string(index=False))

pivot_table = pd.pivot_table(
    df_valid,
    index="n_entities",
    columns="source",
    values="text",
    aggfunc="count",
    fill_value=0
)





In [ ]:
import pandas as pd
from pathlib import Path

INPUT_FILE = "шаг_3_dataset_with_regex_augmented - dataset_with_regex_augmented.csv.csv"
OUTPUT_DIR = Path("test_split_selection")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_SEED = 42

TEST_QUOTAS = {
    1: {
        "synthetic": 9,
        "nerus": 5,
        "factrueval2016": 3,
        "wikipedia": 3,
        "nerel": 4,
        "winer": 3,
        "mokoron": 3,
    },
    2: {
        "synthetic + synthetic": 20,
        "nerus + synthetic": 13,
        "nerus": 13,
        "synthetic": 10,
        "wikipedia + synthetic": 10,
        "factrueval2016 + synthetic": 9,
        "winer + synthetic": 6,
        "nerel + synthetic": 4,
        "wikipedia": 2,
        "mokoron + synthetic": 1,
        "nerel": 1,
        "winer": 1,
        "mokoron": 1,
    },
    3: {
        "nerus + synthetic": 24,
        "synthetic": 20,
        "synthetic + synthetic": 10,
        "wikipedia + synthetic": 6,
        "nerus": 5,
        "winer + synthetic": 4,
        "factrueval2016 + synthetic": 3,
        "nerel + synthetic": 3,
        "mokoron + synthetic": 2,
        "wikipedia": 3,
        "nerel": 1,
    },
    4: {
        "nerus + synthetic": 12,
        "synthetic": 10,
        "synthetic + synthetic": 5,
        "wikipedia + synthetic": 3,
        "nerus": 2,
        "winer + synthetic": 2,
        "factrueval2016 + synthetic": 1,
        "nerel + synthetic": 1,
        "mokoron + synthetic": 1,
        "nerel": 1,
        "wikipedia": 1,
        "winer": 1,
    },
    5: {
        "nerus + synthetic": 14,
        "synthetic + synthetic": 6,
        "synthetic": 3,
        "nerel + synthetic": 2,
        "wikipedia + synthetic": 2,
        "nerus": 1,
        "winer + synthetic": 1,
        "mokoron + synthetic": 1,
        "wikipedia": 1,
    },
    6: {
        "nerus + synthetic": 12,
        "synthetic + synthetic": 4,
        "wikipedia + synthetic": 2,
        "winer + synthetic": 1,
        "nerel + synthetic": 1,
    },
    7: {
        "nerel + synthetic": 3,
        "nerus + synthetic": 1,
        "synthetic + synthetic": 1,
        "wikipedia + synthetic": 1,
    },
    8: {
        "nerus + synthetic": 1,
    }
}

df = pd.read_csv(INPUT_FILE)

required_cols = {"source", "n_entities"}
missing = required_cols - set(df.columns)


df = df.copy()
df.insert(0, "original_row_id", range(len(df)))

df["n_entities"] = pd.to_numeric(df["n_entities"], errors="coerce")
df = df[df["n_entities"].notna()].copy()
df["n_entities"] = df["n_entities"].astype(int)

summary_rows = []
selected_parts = []

for n_ent, source_quota_dict in TEST_QUOTAS.items():
    for source_name, quota in source_quota_dict.items():
        group = df[(df["n_entities"] == n_ent) & (df["source"] == source_name)].copy()
        available = len(group)

        if available < quota:

        sampled = group.sample(n=quota, random_state=RANDOM_SEED)
        selected_parts.append(sampled)

        summary_rows.append({
            "n_entities": n_ent,
            "source": source_name,
            "requested": quota,
            "available": available,
            "selected": len(sampled)
        })

test_df = pd.concat(selected_parts, ignore_index=False).copy()

if test_df["original_row_id"].duplicated().any():
    dup_count = test_df["original_row_id"].duplicated().sum()

train_df = df[~df["original_row_id"].isin(test_df["original_row_id"])].copy()

test_df = test_df.sort_values(by=["n_entities", "source", "original_row_id"]).reset_index(drop=True)
train_df = train_df.sort_values(by=["original_row_id"]).reset_index(drop=True)



In [ ]:
test_path = OUTPUT_DIR / "test_split_300.csv"
train_path = OUTPUT_DIR / "train_pool_1243.csv"
summary_path = OUTPUT_DIR / "test_selection_summary.csv"

test_df.to_csv(test_path, index=False, encoding="utf-8-sig")
train_df.to_csv(train_path, index=False, encoding="utf-8-sig")
pd.DataFrame(summary_rows).to_csv(summary_path, index=False, encoding="utf-8-sig")



In [ ]:
print(f"TEST:  {test_path}")
print(f"TRAIN: {train_path}")
print(f"SUMMARY: {summary_path}")
print()
print(f"Всего в test: {len(test_df)}")
print(f"Всего в train: {len(train_df)}")
print(f"Суммарно: {len(test_df) + len(train_df)}")

Исправление тестовых 300 текстов в DOCCANO

In [ ]:
import pandas as pd
import json
from pathlib import Path

INPUT_CSV = "test_split_selection/test_split_300.csv"
OUTPUT_JSONL = "test_split_selection/test_split_300_for_doccano.jsonl"

SPAN_COLUMN = "typed_entity_spans_regex_augmented_json"
TEXT_COLUMN = "text"

VALID_LABELS = {"PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"}

def safe_load_json(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            try:
                return json.loads(value.replace("'", '"'))
            except Exception:
                return []
    return []

def convert_spans_to_doccano_labels(spans, text):
    labels = []
    seen = set()

    for span in spans:
        if not isinstance(span, dict):
            continue

        start = span.get("start")
        end = span.get("end")
        label = span.get("entity_type")

        if not isinstance(start, int) or not isinstance(end, int):
            continue
        if not isinstance(label, str):
            continue

        label = label.strip().upper()

        if label not in VALID_LABELS:
            continue
        if start < 0 or end <= start or end > len(text):
            continue

        key = (start, end, label)
        if key in seen:
            continue
        seen.add(key)

        labels.append([start, end, label])

    labels.sort(key=lambda x: (x[0], x[1], x[2]))
    return labels

df = pd.read_csv(INPUT_CSV)

required_cols = {TEXT_COLUMN, SPAN_COLUMN}
missing = required_cols - set(df.columns)


records = []

for _, row in df.iterrows():
    text = row[TEXT_COLUMN]
    spans = safe_load_json(row[SPAN_COLUMN])

    if not isinstance(text, str):
        continue

    labels = convert_spans_to_doccano_labels(spans, text)

    record = {
        "text": text,
        "labels": labels
    }
    records.append(record)

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


Эспорт исправленной разметки

In [ ]:
import json
from pathlib import Path
from collections import Counter

INPUT_JSONL = "/content/admin.jsonl 4"
OUTPUT_DIR = Path("prepared_test_gold")
OUTPUT_DIR.mkdir(exist_ok=True)

OUT_JSONL = OUTPUT_DIR / "test_gold_final.jsonl"
OUT_STATS = OUTPUT_DIR / "test_gold_stats.json"

VALID_LABELS = {"PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"}


def load_jsonl(path: str):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
    return records


def normalize_span(span):
    if not isinstance(span, list) or len(span) != 3:
        return None

    start, end, label = span

    if not isinstance(start, int) or not isinstance(end, int):
        return None
    if not isinstance(label, str):
        return None

    label = label.strip().upper()

    if label not in VALID_LABELS:
        return None
    if start < 0 or end <= start:
        return None

    return [start, end, label]


def clean_spans(spans, text):
    cleaned = []
    seen = set()

    for span in spans:
        norm = normalize_span(span)
        if norm is None:
            continue

        start, end, label = norm

        if end > len(text):
            continue

        key = (start, end, label)
        if key in seen:
            continue

        seen.add(key)
        cleaned.append([start, end, label])

    cleaned.sort(key=lambda x: (x[0], x[1], x[2]))
    return cleaned


def merge_doccano_annotation(record):
    manual = record.get("label", [])
    auto = record.get("labels", [])

    manual = manual if isinstance(manual, list) else []
    auto = auto if isinstance(auto, list) else []

    if len(manual) > 0:
        return manual, "manual_label"
    return auto, "auto_labels"


def main():
    records = load_jsonl(INPUT_JSONL)

    output_records = []
    stats = Counter()
    label_counter = Counter()

    for rec in records:
        text = rec.get("text", "")
        rec_id = rec.get("id", None)

        if not isinstance(text, str):
            continue

        chosen_spans, source = merge_doccano_annotation(rec)
        chosen_spans = clean_spans(chosen_spans, text)

        out = {
            "id": rec_id,
            "text": text,
            "label": chosen_spans,
            "annotation_source": source
        }
        output_records.append(out)

        stats["total_records"] += 1
        stats[source] += 1
        stats["total_spans"] += len(chosen_spans)

        if len(chosen_spans) == 0:
            stats["empty_records"] += 1
        else:
            stats["nonempty_records"] += 1

        for _, _, label in chosen_spans:
            label_counter[label] += 1

    with open(OUT_JSONL, "w", encoding="utf-8") as f:
        for rec in output_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    stats_dict = dict(stats)
    stats_dict["label_distribution"] = dict(label_counter)

    with open(OUT_STATS, "w", encoding="utf-8") as f:
        json.dump(stats_dict, f, ensure_ascii=False, indent=2)

    for k, v in stats_dict.items():
        print(f"{k}: {v}")


if __name__ == "__main__":
    main()

Формирование обучающей выборки

In [ ]:
import pandas as pd

FULL_DATASET_PATH = "шаг_3_dataset_with_regex_augmented - dataset_with_regex_augmented.csv.csv"
TEST_SPLIT_PATH = "test_split_selection/test_split_300.csv"

OUTPUT_TRAIN_PATH = "train_pool_1243.csv"
OUTPUT_CHECK_PATH = "train_test_split_check.csv"

full_df = pd.read_csv(FULL_DATASET_PATH)
test_df = pd.read_csv(TEST_SPLIT_PATH)

full_df = full_df.copy()
full_df.insert(0, "original_row_id", range(len(full_df)))


full_ids = set(full_df["original_row_id"].tolist())
test_ids = set(test_df["original_row_id"].tolist())

missing_ids = test_ids - full_ids

train_df = full_df[~full_df["original_row_id"].isin(test_ids)].copy()

train_df = train_df.sort_values(by="original_row_id").reset_index(drop=True)


train_ids = set(train_df["original_row_id"].tolist())
intersection = train_ids & test_ids

train_df.to_csv(OUTPUT_TRAIN_PATH, index=False, encoding="utf-8-sig")

check_df = pd.DataFrame({
    "dataset_part": ["full", "test", "train"],
    "count_rows": [len(full_df), len(test_df), len(train_df)]
})
check_df.to_csv(OUTPUT_CHECK_PATH, index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd

INPUT_FILE = "train_pool_1243.csv"

OUTPUT_500 = "train_subset_500.csv"
OUTPUT_REST = "train_remaining_743.csv"

RANDOM_SEED = 42

df = pd.read_csv(INPUT_FILE)

df_shuffled = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

subset_500 = df_shuffled.iloc[:500].copy()
remaining = df_shuffled.iloc[500:].copy()

subset_500.to_csv(OUTPUT_500, index=False, encoding="utf-8-sig")
remaining.to_csv(OUTPUT_REST, index=False, encoding="utf-8-sig")


Формат для doccano

In [ ]:
import pandas as pd
import json
from pathlib import Path

INPUT_CSV = "train_subset_500.csv"
OUTPUT_JSONL = "train_subset_500_for_doccano.jsonl"

TEXT_COLUMN = "text"
SPANS_COLUMN = "typed_entity_spans_regex_augmented_json"

VALID_LABELS = {"PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"}


def safe_load_json(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            try:
                return json.loads(value.replace("'", '"'))
            except Exception:
                return []
    return []


def convert_spans_to_doccano_labels(spans, text):

    labels = []
    seen = set()

    for span in spans:
        if not isinstance(span, dict):
            continue

        start = span.get("start")
        end = span.get("end")
        label = span.get("entity_type")

        if not isinstance(start, int) or not isinstance(end, int):
            continue
        if not isinstance(label, str):
            continue

        label = label.strip().upper()

        if label not in VALID_LABELS:
            continue
        if start < 0 or end <= start or end > len(text):
            continue

        key = (start, end, label)
        if key in seen:
            continue
        seen.add(key)

        labels.append([start, end, label])

    labels.sort(key=lambda x: (x[0], x[1], x[2]))
    return labels

df = pd.read_csv(INPUT_CSV)

required_cols = {TEXT_COLUMN, SPANS_COLUMN}
missing = required_cols - set(df.columns)

records = []

for _, row in df.iterrows():
    text = row[TEXT_COLUMN]
    spans = safe_load_json(row[SPANS_COLUMN])

    if not isinstance(text, str):
        continue

    labels = convert_spans_to_doccano_labels(spans, text)

    record = {
        "text": text,
        "labels": labels,
        "meta": {
            "original_row_id": int(row["original_row_id"]) if "original_row_id" in row and pd.notna(row["original_row_id"]) else None,
            "source": row["source"] if "source" in row and pd.notna(row["source"]) else None,
            "n_entities": int(row["n_entities"]) if "n_entities" in row and pd.notna(row["n_entities"]) else None
        }
    }

    records.append(record)

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")



Приведение к нужному виду и обучение pre-annotation модели BERT

In [ ]:
import json
from pathlib import Path
from collections import Counter

INPUT_JSONL = "/content/admin.jsonl 6"
OUTPUT_DIR = Path("prepared_train_gold_500")
OUTPUT_DIR.mkdir(exist_ok=True)

OUT_JSONL = OUTPUT_DIR / "train_gold_500_final.jsonl"
OUT_STATS = OUTPUT_DIR / "train_gold_500_stats.json"

VALID_LABELS = {"PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"}

def load_jsonl(path: str):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continueы
    return records


def normalize_span(span):
    if not isinstance(span, list) or len(span) != 3:
        return None

    start, end, label = span

    if not isinstance(start, int) or not isinstance(end, int):
        return None
    if not isinstance(label, str):
        return None

    label = label.strip().upper()

    if label not in VALID_LABELS:
        return None
    if start < 0 or end <= start:
        return None

    return [start, end, label]


def clean_spans(spans, text):
    cleaned = []
    seen = set()

    for span in spans:
        norm = normalize_span(span)
        if norm is None:
            continue

        start, end, label = norm

        if end > len(text):
            continue

        key = (start, end, label)
        if key in seen:
            continue

        seen.add(key)
        cleaned.append([start, end, label])

    cleaned.sort(key=lambda x: (x[0], x[1], x[2]))
    return cleaned


def main():
    records = load_jsonl(INPUT_JSONL)

    output_records = []
    stats = Counter()
    label_counter = Counter()
    source_counter = Counter()
    n_entities_counter = Counter()

    for rec in records:
        text = rec.get("text", "")
        meta = rec.get("meta", {}) if isinstance(rec.get("meta", {}), dict) else {}
        spans = rec.get("label", [])

        if not isinstance(text, str):
            continue
        if not isinstance(spans, list):
            spans = []

        cleaned_spans = clean_spans(spans, text)

        source = meta.get("source")
        n_entities = meta.get("n_entities")
        original_row_id = meta.get("original_row_id")

        out = {
            "text": text,
            "label": cleaned_spans,
            "meta": {
                "original_row_id": original_row_id,
                "source": source,
                "n_entities": n_entities
            }
        }

        output_records.append(out)

        stats["total_records"] += 1
        stats["total_spans"] += len(cleaned_spans)

        if len(cleaned_spans) == 0:
            stats["empty_records"] += 1
        else:
            stats["nonempty_records"] += 1

        if source is not None:
            source_counter[str(source)] += 1
        if n_entities is not None:
            n_entities_counter[str(n_entities)] += 1

        for _, _, label in cleaned_spans:
            label_counter[label] += 1

    with open(OUT_JSONL, "w", encoding="utf-8") as f:
        for rec in output_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    stats_dict = dict(stats)
    stats_dict["label_distribution"] = dict(label_counter)
    stats_dict["source_distribution"] = dict(source_counter)
    stats_dict["n_entities_distribution"] = dict(n_entities_counter)

    with open(OUT_STATS, "w", encoding="utf-8") as f:
        json.dump(stats_dict, f, ensure_ascii=False, indent=2)

    for k, v in stats_dict.items():
        print(f"{k}: {v}")


if __name__ == "__main__":
    main()

Обучение BERT

In [ ]:
!pip install evaluate seqeval

In [ ]:
import json
import random
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)
import evaluate

INPUT_JSONL = "/content/prepared_train_gold_500/train_gold_500_final.jsonl"
OUTPUT_DIR = Path("bert_preannotation_v0")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

MODEL_NAME = "DeepPavlov/rubert-base-cased"
MAX_LENGTH = 256
SEED = 42

TRAIN_RATIO = 0.8
TEST_RATIO = 0.2

ENTITY_TYPES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]

LABEL_LIST = ["O"]
for ent in ENTITY_TYPES:
    LABEL_LIST.append(f"B-{ent}")
    LABEL_LIST.append(f"I-{ent}")

label2id = {label: i for i, label in enumerate(LABEL_LIST)}
id2label = {i: label for label, i in label2id.items()}

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

seqeval = evaluate.load("seqeval")


def load_jsonl(path: str) -> List[Dict[str, Any]]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
    return records


def clean_spans(spans: List[List[Any]], text: str) -> List[Dict[str, Any]]:
    cleaned = []
    seen = set()

    if not isinstance(spans, list):
        return cleaned

    for span in spans:
        if not isinstance(span, list) or len(span) != 3:
            continue

        start, end, label = span

        if not isinstance(start, int) or not isinstance(end, int) or not isinstance(label, str):
            continue
        if label not in ENTITY_TYPES:
            continue
        if start < 0 or end <= start or end > len(text):
            continue

        key = (start, end, label)
        if key in seen:
            continue
        seen.add(key)

        cleaned.append({
            "start": start,
            "end": end,
            "label": label
        })

    cleaned.sort(key=lambda x: (x["start"], x["end"], x["label"]))

    no_overlap = []
    last_end = -1
    for span in cleaned:
        if span["start"] < last_end:
            continue
        no_overlap.append(span)
        last_end = span["end"]

    return no_overlap


In [ ]:
records = load_jsonl(INPUT_JSONL)

examples = []
for rec in records:
    text = rec.get("text", "")
    spans = rec.get("label", [])
    meta = rec.get("meta", {}) if isinstance(rec.get("meta", {}), dict) else {}

    if not isinstance(text, str):
        continue

    clean = clean_spans(spans, text)

    examples.append({
        "text": text,
        "spans": clean,
        "meta_source": meta.get("source"),
        "meta_n_entities": meta.get("n_entities"),
        "meta_original_row_id": meta.get("original_row_id"),
    })




In [ ]:
random.shuffle(examples)
n_total = len(examples)
n_train = int(n_total * TRAIN_RATIO)

train_examples = examples[:n_train]
test_examples = examples[n_train:]

print(f"Train: {len(train_examples)}")
print(f"Test:  {len(test_examples)}")


def to_dataset_dict(records_split: List[Dict[str, Any]]) -> Dataset:
    return Dataset.from_list(records_split)


raw_datasets = DatasetDict({
    "train": to_dataset_dict(train_examples),
    "test": to_dataset_dict(test_examples),
})

for split_name, split_records in [("train", train_examples), ("test", test_examples)]:
    out_path = OUTPUT_DIR / f"{split_name}_split.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for rec in split_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def tokenize_and_align_labels(batch):
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )

    all_labels = []

    for i, offsets in enumerate(tokenized["offset_mapping"]):
        spans = batch["spans"][i]
        labels = []

        for token_start, token_end in offsets:
            if token_start == 0 and token_end == 0:
                labels.append(-100)
                continue

            assigned = "O"

            for span in spans:
                span_start = span["start"]
                span_end = span["end"]
                span_label = span["label"]

                if token_end <= span_start or token_start >= span_end:
                    continue

                if token_start == span_start:
                    assigned = f"B-{span_label}"
                else:
                    assigned = f"I-{span_label}"
                break

            labels.append(label2id[assigned])

        all_labels.append(labels)

    tokenized["labels"] = all_labels
    tokenized.pop("offset_mapping")
    return tokenized


tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)






In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        pred_labels = []
        gold_labels = []

        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue
            pred_labels.append(LABEL_LIST[pred_id])
            gold_labels.append(LABEL_LIST[label_id])

        true_predictions.append(pred_labels)
        true_labels.append(gold_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    metrics = {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

    for entity_type in ENTITY_TYPES:
        if entity_type in results:
            metrics[f"{entity_type}_precision"] = results[entity_type]["precision"]
            metrics[f"{entity_type}_recall"] = results[entity_type]["recall"]
            metrics[f"{entity_type}_f1"] = results[entity_type]["f1"]

    return metrics

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

metrics = trainer.evaluate(tokenized_datasets["test"])


In [ ]:
final_model_dir = OUTPUT_DIR / "final_model"
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

with open(OUTPUT_DIR / "label_mappings.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "label_list": LABEL_LIST,
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()},
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "train_size": len(train_examples),
            "test_size": len(test_examples),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(OUTPUT_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

In [ ]:
tokenized_datasets.save_to_disk(str(OUTPUT_DIR / "tokenized_dataset"))


In [ ]:
import shutil

shutil.make_archive("bert_preannotation_v0", "zip", "bert_preannotation_v0")

In [ ]:
from google.colab import files

files.download("bert_preannotation_v0.zip")

Подготовка файла с 743 текстами

In [ ]:
import pandas as pd
import json
from pathlib import Path

INPUT_CSV = "/train_remaining_743.csv"

OUTPUT_DIR = Path("prepared_inference_743")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

OUT_JSONL = OUTPUT_DIR / "train_remaining_743_for_inference.jsonl"
OUT_CSV = OUTPUT_DIR / "train_remaining_743_for_inference.csv"
OUT_STATS = OUTPUT_DIR / "train_remaining_743_inference_stats.json"

REQUIRED_COLUMNS = ["original_row_id", "source", "text", "n_entities"]


df = pd.read_csv(INPUT_CSV)

missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]


def clean_text(value):
    if pd.isna(value):
        return ""
    text = str(value)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text.strip()


records = []
bad_rows = 0

for _, row in df.iterrows():
    text = clean_text(row["text"])

    if not text:
        bad_rows += 1
        continue

    original_row_id = row["original_row_id"]
    source = row["source"]
    n_entities = row["n_entities"]

    meta = {
        "original_row_id": int(original_row_id) if pd.notna(original_row_id) else None,
        "source": str(source) if pd.notna(source) else None,
        "n_entities": int(n_entities) if pd.notna(n_entities) else None,
    }

    records.append({
        "text": text,
        "meta": meta
    })



In [ ]:
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


light_df = pd.DataFrame([
    {
        "original_row_id": rec["meta"]["original_row_id"],
        "source": rec["meta"]["source"],
        "n_entities": rec["meta"]["n_entities"],
        "text": rec["text"],
    }
    for rec in records
])

light_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

In [ ]:
source_counts = light_df["source"].value_counts(dropna=False).to_dict()
n_entities_counts = light_df["n_entities"].value_counts(dropna=False).sort_index().to_dict()

stats = {
    "total_input_rows": int(len(df)),
    "total_output_records": int(len(records)),
    "dropped_empty_text_rows": int(bad_rows),
    "source_distribution": source_counts,
    "n_entities_distribution": {str(k): int(v) for k, v in n_entities_counts.items()},
}

with open(OUT_STATS, "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)


In [ ]:
import pandas as pd
import json
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

INPUT_CSV = "/train_remaining_743.csv"
MODEL_DIR = "bert_preannotation_v0/final_model"

OUTPUT_DIR = Path("bert_preannotated_743")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

OUT_DOCCANO_JSONL = OUTPUT_DIR / "train_remaining_743_for_doccano_with_bert.jsonl"
OUT_ANALYSIS_JSONL = OUTPUT_DIR / "bert_preannotation_analysis.jsonl"
OUT_STATS_JSON = OUTPUT_DIR / "bert_preannotation_stats.json"

TEXT_COLUMN = "text"
EXISTING_SPANS_COLUMN = "typed_entity_spans_regex_augmented_json"

VALID_LABELS = {"PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"}

SCORE_THRESHOLD = 0.60

device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)

bert_ner = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=device,
)

def safe_load_json(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            try:
                return json.loads(value.replace("'", '"'))
            except Exception:
                return []
    return []


def normalize_existing_spans(spans, text):
    cleaned = []
    seen = set()

    for span in spans:
        if not isinstance(span, dict):
            continue

        start = span.get("start")
        end = span.get("end")
        label = span.get("entity_type")

        if not isinstance(start, int) or not isinstance(end, int):
            continue
        if not isinstance(label, str):
            continue

        label = label.strip().upper()

        if label not in VALID_LABELS:
            continue
        if start < 0 or end <= start or end > len(text):
            continue

        key = (start, end, label)
        if key in seen:
            continue
        seen.add(key)

        cleaned.append({
            "start": start,
            "end": end,
            "label": label,
            "text": text[start:end],
            "source": "existing"
        })

    cleaned.sort(key=lambda x: (x["start"], x["end"], x["label"]))
    return cleaned


def normalize_bert_label(label):
    if label in VALID_LABELS:
        return label
    if isinstance(label, str) and (label.startswith("B-") or label.startswith("I-")):
        return label[2:]
    return label


def predict_bert_spans(text):
    preds = bert_ner(text)
    spans = []
    seen = set()

    for pred in preds:
        label = pred.get("entity_group", pred.get("entity", ""))
        label = normalize_bert_label(label)

        if label not in VALID_LABELS:
            continue

        score = float(pred.get("score", 0.0))
        if score < SCORE_THRESHOLD:
            continue

        start = int(pred["start"])
        end = int(pred["end"])

        if start < 0 or end <= start or end > len(text):
            continue

        key = (start, end, label)
        if key in seen:
            continue
        seen.add(key)

        spans.append({
            "start": start,
            "end": end,
            "label": label,
            "text": text[start:end],
            "score": score,
            "source": "bert"
        })

    spans.sort(key=lambda x: (x["start"], x["end"], x["label"]))
    return spans


def spans_overlap(a, b):
    return not (a["end"] <= b["start"] or a["start"] >= b["end"])


def merge_existing_and_bert(existing_spans, bert_spans):

    merged = existing_spans.copy()
    conflicts = []
    added = []
    duplicates = []

    existing_keys = {(s["start"], s["end"], s["label"]) for s in existing_spans}

    for pred in bert_spans:
        key = (pred["start"], pred["end"], pred["label"])

        if key in existing_keys:
            duplicates.append(pred)
            continue

        overlapping = [ex for ex in existing_spans if spans_overlap(pred, ex)]
        if overlapping:
            conflicts.append({
                "bert_span": pred,
                "existing_spans": overlapping
            })
            continue

        merged.append(pred)
        added.append(pred)

    merged.sort(key=lambda x: (x["start"], x["end"], x["label"]))
    return merged, added, duplicates, conflicts


def to_doccano_labels(spans):
    return [[s["start"], s["end"], s["label"]] for s in spans]


df = pd.read_csv(INPUT_CSV)

required_cols = {"original_row_id", "source", "text", "n_entities", EXISTING_SPANS_COLUMN}
missing = required_cols - set(df.columns)


doccano_records = []
analysis_records = []

total_existing = 0
total_bert_pred = 0
total_added = 0
total_duplicates = 0
total_conflicts = 0

for i, row in df.iterrows():
    text = str(row[TEXT_COLUMN]) if pd.notna(row[TEXT_COLUMN]) else ""
    if not text.strip():
        continue

    existing_raw = safe_load_json(row[EXISTING_SPANS_COLUMN])
    existing_spans = normalize_existing_spans(existing_raw, text)
    bert_spans = predict_bert_spans(text)

    merged_spans, added_spans, duplicate_spans, conflicts = merge_existing_and_bert(
        existing_spans, bert_spans
    )

    total_existing += len(existing_spans)
    total_bert_pred += len(bert_spans)
    total_added += len(added_spans)
    total_duplicates += len(duplicate_spans)
    total_conflicts += len(conflicts)

    doccano_record = {
        "text": text,
        "labels": to_doccano_labels(merged_spans),
        "meta": {
            "original_row_id": int(row["original_row_id"]) if pd.notna(row["original_row_id"]) else None,
            "source": str(row["source"]) if pd.notna(row["source"]) else None,
            "n_entities": int(row["n_entities"]) if pd.notna(row["n_entities"]) else None,
            "existing_span_count": len(existing_spans),
            "bert_pred_count": len(bert_spans),
            "bert_added_count": len(added_spans),
            "duplicate_count": len(duplicate_spans),
            "conflict_count": len(conflicts),
        }
    }
    doccano_records.append(doccano_record)

    analysis_records.append({
        "text": text,
        "meta": doccano_record["meta"],
        "existing_spans": existing_spans,
        "bert_spans": bert_spans,
        "added_spans": added_spans,
        "duplicate_spans": duplicate_spans,
        "conflicts": conflicts,
        "final_labels": doccano_record["labels"],
    })


with open(OUT_DOCCANO_JSONL, "w", encoding="utf-8") as f:
    for rec in doccano_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

with open(OUT_ANALYSIS_JSONL, "w", encoding="utf-8") as f:
    for rec in analysis_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

stats = {
    "total_records": len(doccano_records),
    "total_existing_spans": total_existing,
    "total_bert_predicted_spans": total_bert_pred,
    "total_bert_added_spans": total_added,
    "total_duplicate_spans": total_duplicates,
    "total_conflicts": total_conflicts,
    "score_threshold": SCORE_THRESHOLD,
    "model_dir": MODEL_DIR,
}

with open(OUT_STATS_JSON, "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)


In [ ]:
import json
from pathlib import Path

INPUT_ANALYSIS_JSONL = "bert_preannotated_743/bert_preannotation_analysis.jsonl"
OUTPUT_DIR = Path("doccano_ready_from_bert_analysis")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

LABEL_SOURCE = "bert_spans"

OUTPUT_JSONL = OUTPUT_DIR / f"train_remaining_743_{LABEL_SOURCE}_for_doccano.jsonl"

VALID_LABELS = {"PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"}


def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

    return records


def normalize_span_list(spans, text):
    labels = []
    seen = set()

    if not isinstance(spans, list):
        return labels

    for span in spans:
        if isinstance(span, dict):
            start = span.get("start")
            end = span.get("end")
            label = span.get("label")
        elif isinstance(span, (list, tuple)) and len(span) == 3:
            start, end, label = span
        else:
            continue

        if not isinstance(start, int) or not isinstance(end, int):
            continue
        if not isinstance(label, str):
            continue

        label = label.strip().upper()

        if label not in VALID_LABELS:
            continue
        if start < 0 or end <= start or end > len(text):
            continue

        key = (start, end, label)
        if key in seen:
            continue
        seen.add(key)

        labels.append([start, end, label])

    labels.sort(key=lambda x: (x[0], x[1], x[2]))
    return labels


records = load_jsonl(INPUT_ANALYSIS_JSONL)


doccano_records = []

for rec in records:
    text = rec.get("text", "")
    if not isinstance(text, str) or not text.strip():
        continue

    meta = rec.get("meta", {})
    if not isinstance(meta, dict):
        meta = {}

    if LABEL_SOURCE not in rec:
        available = list(rec.keys())

    spans = rec.get(LABEL_SOURCE, [])
    labels = normalize_span_list(spans, text)

    out = {
        "text": text,
        "labels": labels,
        "meta": {
            "original_row_id": meta.get("original_row_id"),
            "source": meta.get("source"),
            "n_entities": meta.get("n_entities"),
            "existing_span_count": meta.get("existing_span_count"),
            "bert_pred_count": meta.get("bert_pred_count"),
            "bert_added_count": meta.get("bert_added_count"),
            "duplicate_count": meta.get("duplicate_count"),
            "conflict_count": meta.get("conflict_count"),
            "label_source": LABEL_SOURCE,
        }
    }

    doccano_records.append(out)

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in doccano_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


In [ ]:
from google.colab import files

files.download("doccano_ready_from_bert_analysis/train_remaining_743_bert_spans_for_doccano.jsonl")

Сбор обучающей выборки (500+743)

In [ ]:
import json
from pathlib import Path
from collections import Counter

INPUT_JSONL = "/content/admin.jsonl 7"

OUTPUT_DIR = Path("prepared_train_743_final")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

OUT_JSONL = OUTPUT_DIR / "train_743_final.jsonl"
OUT_STATS = OUTPUT_DIR / "train_743_final_stats.json"

VALID_LABELS = {"PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"}


def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

    return records


def normalize_span(span, text):
    if not isinstance(span, list) or len(span) != 3:
        return None

    start, end, label = span

    if not isinstance(start, int) or not isinstance(end, int):
        return None
    if not isinstance(label, str):
        return None

    label = label.strip().upper()

    if label not in VALID_LABELS:
        return None

    if start < 0 or end <= start or end > len(text):
        return None

    return [start, end, label]


def clean_spans(spans, text):
    cleaned = []
    seen = set()

    if not isinstance(spans, list):
        return cleaned

    for span in spans:
        norm = normalize_span(span, text)
        if norm is None:
            continue

        start, end, label = norm
        key = (start, end, label)

        if key in seen:
            continue

        seen.add(key)
        cleaned.append([start, end, label])

    cleaned.sort(key=lambda x: (x[0], x[1], x[2]))

    no_overlap = []
    last_end = -1

    for start, end, label in cleaned:
        if start < last_end:
            continue
        no_overlap.append([start, end, label])
        last_end = end

    return no_overlap


def choose_final_labels(record):
    manual = record.get("label", [])
    original = record.get("labels", [])

    manual = manual if isinstance(manual, list) else []
    original = original if isinstance(original, list) else []

    if len(manual) > 0:
        return manual, "manual_label"

    return original, "original_labels"


records = load_jsonl(INPUT_JSONL)

output_records = []
stats = Counter()
label_counter = Counter()
source_counter = Counter()

for rec in records:
    text = rec.get("text", "")

    if not isinstance(text, str) or not text.strip():
        continue

    chosen_spans, annotation_source = choose_final_labels(rec)
    final_labels = clean_spans(chosen_spans, text)

    meta = rec.get("meta", {})
    if not isinstance(meta, dict):
        meta = {}

    out = {
        "text": text,
        "label": final_labels,
        "annotation_source": annotation_source,
        "meta": {
            "original_row_id": meta.get("original_row_id"),
            "source": meta.get("source"),
            "n_entities": meta.get("n_entities"),
        }
    }

    output_records.append(out)

    stats["total_records"] += 1
    stats[annotation_source] += 1
    stats["total_spans"] += len(final_labels)

    if len(final_labels) == 0:
        stats["empty_records"] += 1
    else:
        stats["nonempty_records"] += 1

    source = meta.get("source")
    if source is not None:
        source_counter[str(source)] += 1

    for _, _, label in final_labels:
        label_counter[label] += 1


with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for rec in output_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

stats_dict = dict(stats)
stats_dict["label_distribution"] = dict(label_counter)
stats_dict["source_distribution"] = dict(source_counter)

with open(OUT_STATS, "w", encoding="utf-8") as f:
    json.dump(stats_dict, f, ensure_ascii=False, indent=2)


for k, v in stats_dict.items():
    print(f"{k}: {v}")

In [ ]:
import json
from pathlib import Path
from collections import Counter

TRAIN_500_JSONL = "/content/train_gold_500_final.jsonl"
TRAIN_743_JSONL = "/content/prepared_train_743_final/train_743_final.jsonl"

OUTPUT_DIR = Path("prepared_train_final_1243")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

OUT_JSONL = OUTPUT_DIR / "train_final_1243.jsonl"
OUT_CSV = OUTPUT_DIR / "train_final_1243.csv"
OUT_STATS = OUTPUT_DIR / "train_final_1243_stats.json"

VALID_LABELS = {"PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"}


def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def clean_labels(labels, text):
    cleaned = []
    seen = set()

    if not isinstance(labels, list):
        return cleaned

    for span in labels:
        if not isinstance(span, list) or len(span) != 3:
            continue

        start, end, label = span
        if not isinstance(start, int) or not isinstance(end, int):
            continue
        if not isinstance(label, str):
            continue

        label = label.strip().upper()

        if label not in VALID_LABELS:
            continue
        if start < 0 or end <= start or end > len(text):
            continue

        key = (start, end, label)
        if key in seen:
            continue

        seen.add(key)
        cleaned.append([start, end, label])

    cleaned.sort(key=lambda x: (x[0], x[1], x[2]))
    return cleaned


def normalize_record(rec, subset_name):
    text = rec.get("text", "")
    labels = rec.get("label", [])

    meta = rec.get("meta", {})
    if not isinstance(meta, dict):
        meta = {}

    source = meta.get("source")
    original_row_id = meta.get("original_row_id")
    n_entities = meta.get("n_entities")

    labels = clean_labels(labels, text)

    return {
        "text": text,
        "label": labels,
        "source": source,
        "original_row_id": original_row_id,
        "n_entities": n_entities,
        "subset": subset_name,
        "meta": {
            "original_row_id": original_row_id,
            "source": source,
            "n_entities": n_entities,
            "subset": subset_name,
        }
    }


records_500 = load_jsonl(TRAIN_500_JSONL)
records_743 = load_jsonl(TRAIN_743_JSONL)

final_records = []
final_records += [normalize_record(r, "train_gold_500") for r in records_500]
final_records += [normalize_record(r, "train_corrected_743") for r in records_743]



In [ ]:
seen_ids = set()
deduped = []
duplicates = []

for rec in final_records:
    rid = rec.get("original_row_id")
    if rid is not None and rid in seen_ids:
        duplicates.append(rid)
        continue
    if rid is not None:
        seen_ids.add(rid)
    deduped.append(rec)

final_records = deduped

final_records.sort(
    key=lambda x: (
        10**12 if x.get("original_row_id") is None else int(x["original_row_id"])
    )
)

In [ ]:
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for rec in final_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

import pandas as pd

csv_rows = []
for rec in final_records:
    csv_rows.append({
        "original_row_id": rec["original_row_id"],
        "source": rec["source"],
        "n_entities": rec["n_entities"],
        "subset": rec["subset"],
        "text": rec["text"],
        "label": json.dumps(rec["label"], ensure_ascii=False),
        "n_labels": len(rec["label"]),
    })

df = pd.DataFrame(csv_rows)
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

In [ ]:
stats = Counter()
label_counter = Counter()
source_counter = Counter()
subset_counter = Counter()

In [ ]:
for rec in final_records:
    stats["total_records"] += 1
    stats["total_spans"] += len(rec["label"])

    if len(rec["label"]) == 0:
        stats["empty_records"] += 1
    else:
        stats["nonempty_records"] += 1

    source_counter[str(rec["source"])] += 1
    subset_counter[str(rec["subset"])] += 1

    for _, _, label in rec["label"]:
        label_counter[label] += 1

stats_dict = dict(stats)
stats_dict["label_distribution"] = dict(label_counter)
stats_dict["source_distribution"] = dict(source_counter)
stats_dict["subset_distribution"] = dict(subset_counter)
stats_dict["duplicate_original_row_ids_removed"] = duplicates

with open(OUT_STATS, "w", encoding="utf-8") as f:
    json.dump(stats_dict, f, ensure_ascii=False, indent=2)


In [ ]:
from google.colab import files
files.download("prepared_train_final_1243/train_final_1243.csv")

In [ ]:
import pandas as pd
from pathlib import Path

INPUT_CSV = "prepared_train_final_1243/train_final_1243.csv"

OUTPUT_DIR = Path("prepared_train_final_1243_v2")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

OUTPUT_CSV = OUTPUT_DIR / "train_final_1243_v2.csv"

df = pd.read_csv(INPUT_CSV)

cols_to_drop = ["original_row_id", "n_entities", "subset"]
df_v2 = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

df_v2["source"] = df_v2["source"].replace({
    "synthetic + synthetic": "synthetic"
})

df_v2.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(df_v2["source"].value_counts())

In [ ]:
from google.colab import files

files.download("prepared_train_final_1243_v2/train_final_1243_v2.csv")

In [ ]:
import pandas as pd
import json
from pathlib import Path
from collections import Counter

TEST_JSONL = "test_gold_final.jsonl"
TEST_SOURCE_CSV = "/content/test_split_300.csv"

OUTPUT_DIR = Path("prepared_test_final_300_v3")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

OUT_CSV = OUTPUT_DIR / "test_final_300_v3.csv"
OUT_JSONL = OUTPUT_DIR / "test_final_300_v3.jsonl"
OUT_STATS = OUTPUT_DIR / "test_final_300_v3_stats.json"

VALID_LABELS = {"PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"}

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def clean_labels(labels, text):
    cleaned = []
    seen = set()

    if not isinstance(labels, list):
        return cleaned

    for span in labels:
        if not isinstance(span, list) or len(span) != 3:
            continue

        start, end, label = span

        if not isinstance(start, int) or not isinstance(end, int):
            continue
        if not isinstance(label, str):
            continue

        label = label.strip().upper()

        if label not in VALID_LABELS:
            continue
        if start < 0 or end <= start or end > len(text):
            continue

        key = (start, end, label)
        if key in seen:
            continue

        seen.add(key)
        cleaned.append([start, end, label])

    cleaned.sort(key=lambda x: (x[0], x[1], x[2]))
    return cleaned


def choose_final_labels(rec):
    manual = rec.get("label", [])
    original = rec.get("labels", [])

    manual = manual if isinstance(manual, list) else []
    original = original if isinstance(original, list) else []

    if len(manual) > 0:
        return manual

    return original

source_df = pd.read_csv(TEST_SOURCE_CSV)

required_cols = {"text", "source", "original_row_id", "n_entities"}
missing = required_cols - set(source_df.columns)


text_to_meta = {}

for _, row in source_df.iterrows():
    text = str(row["text"])

    text_to_meta[text] = {
        "source": row["source"],
        "original_row_id": int(row["original_row_id"]) if pd.notna(row["original_row_id"]) else None,
        "n_entities": int(row["n_entities"]) if pd.notna(row["n_entities"]) else None,
    }


records = load_jsonl(TEST_JSONL)

final_records = []
stats = Counter()
label_counter = Counter()
source_counter = Counter()
not_found = 0

for rec in records:
    text = rec.get("text", "")

    if not isinstance(text, str) or not text.strip():
        continue

    meta_from_source = text_to_meta.get(text)

    if meta_from_source is None:
        not_found += 1
        source = None
        original_row_id = None
        n_entities = None
    else:
        source = meta_from_source["source"]
        original_row_id = meta_from_source["original_row_id"]
        n_entities = meta_from_source["n_entities"]

    final_labels = clean_labels(choose_final_labels(rec), text)

    out = {
        "text": text,
        "label": final_labels,
        "source": source,
        "original_row_id": original_row_id,
        "n_entities": n_entities,
    }

    final_records.append(out)

    stats["total_records"] += 1
    stats["total_spans"] += len(final_labels)

    if source is None:
        stats["source_not_found"] += 1
    else:
        source_counter[str(source)] += 1

    for _, _, label in final_labels:
        label_counter[label] += 1


with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for rec in final_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


csv_rows = []

for rec in final_records:
    csv_rows.append({
        "original_row_id": rec["original_row_id"],
        "source": rec["source"],
        "n_entities": rec["n_entities"],
        "text": rec["text"],
        "label": json.dumps(rec["label"], ensure_ascii=False),
        "n_labels": len(rec["label"]),
    })

df = pd.DataFrame(csv_rows)
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")


stats_dict = dict(stats)
stats_dict["label_distribution"] = dict(label_counter)
stats_dict["source_distribution"] = dict(source_counter)

with open(OUT_STATS, "w", encoding="utf-8") as f:
    json.dump(stats_dict, f, ensure_ascii=False, indent=2)

print(df["source"].value_counts(dropna=False))

In [ ]:
from google.colab import files

files.download("prepared_test_final_300_v3/test_final_300_v3.csv")